# Pertemuan 10 - Algoritma Klasifikasi (Bagian 2)
Nama : Abdul Zaki Al-Muttaqin
Nim : 240401010042
Kelas : IF403

## Tujuan Praktikum
Mempelajari penerapan algoritma Random Forest untuk klasifikasi pada kasus Customer Churn, memahami penanganan imbalanced dataset menggunakan class_weight, serta mengevaluasi model menggunakan Precision, Recall, F1-Score, dan ROC-AUC.

In [ ]:
# Load dan Eksplorasi Data
import pandas as pd

# Membaca dataset
df = pd.read_csv("Telco-Customer-Churn.csv")

# Menampilkan ukuran dataset
print("Shape:", df.shape)

# Menampilkan 5 data pertama
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# Melihat Informasi Dataset
print("Informasi Dataset:")
df.info()

print("\nMissing Value:")
print(df.isnull().sum())

print("\nTipe Data:")
print(df.dtypes)

Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 

In [ ]:
# Mengecek Distribusi Target
print("Distribusi Target (Jumlah):")
print(df["Churn"].value_counts())

print("\nDistribusi Target (Persentase):")
print(df["Churn"].value_counts(normalize=True).round(3))

Distribusi Target (Jumlah):
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Distribusi Target (Persentase):
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


In [ ]:
# Menghapus kolom customerID karena hanya sebagai identitas
df = df.drop("customerID", axis=1)

# Mengubah TotalCharges menjadi numerik
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Mengisi nilai kosong dengan median
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Mengubah target Churn menjadi angka
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [ ]:
# One-Hot Encoding untuk seluruh fitur kategorikal
df = pd.get_dummies(df, drop_first=True)

print("Shape setelah encoding:", df.shape)

print("\n5 Kolom Pertama:")
print(df.columns[:15])

Shape setelah encoding: (7043, 31)

5 Kolom Pertama:
Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn',
       'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes'],
      dtype='object')


In [ ]:
# Membagi Data(Train Test Split)
from sklearn.model_selection import train_test_split

# Memisahkan fitur dan target
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Membagi data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Shape X:", X.shape)
print("Shape y:", y.shape)
print("\nTrain:", X_train.shape)
print("Test :", X_test.shape)

Shape X: (7043, 30)
Shape y: (7043,)

Train: (5634, 30)
Test : (1409, 30)


In [ ]:
# Membangun Model Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

print("Model Random Forest berhasil dilatih!")

Model Random Forest berhasil dilatih!


In [ ]:
# Prediksi data testing
y_pred = rf_model.predict(X_test)

print("5 Prediksi Pertama:")
print(y_pred[:5])

print("\n5 Data Asli:")
print(y_test[:5].values)

5 Prediksi Pertama:
[0 1 0 0 0]

5 Data Asli:
[0 0 0 0 0]


In [ ]:
# Evaluasi Model
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.7877927608232789
Precision: 0.6271186440677966
Recall   : 0.4946524064171123
F1 Score : 0.5530642750373692
ROC AUC  : 0.6941861065901986

Confusion Matrix:
[[925 110]
 [189 185]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.49      0.55       374

    accuracy                           0.79      1409
   macro avg       0.73      0.69      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [ ]:
# Probabilitas Churn
y_prob = rf_model.predict_proba(X_test)

print("Probabilitas 5 pelanggan pertama:")
print(y_prob[:5])

Probabilitas 5 pelanggan pertama:
[[1.   0.  ]
 [0.24 0.76]
 [0.93 0.07]
 [0.66 0.34]
 [1.   0.  ]]


# Kesimpulan

Pada praktikum ini saya mempelajari penggunaan Random Forest untuk melakukan klasifikasi pada dataset Customer Churn. Saya juga mempelajari penanganan dataset yang tidak seimbang serta penggunaan Accuracy, Precision, Recall, F1-Score, dan ROC-AUC untuk mengevaluasi model.

Hasil evaluasi menunjukkan bahwa model memperoleh accuracy sebesar 78,78%, precision sebesar 62,71%, recall sebesar 49,46%, F1-Score sebesar 55,31%, dan ROC-AUC sebesar 69,42%. Hasil tersebut menunjukkan bahwa model masih memiliki keterbatasan dalam mengenali pelanggan yang benar-benar akan churn, terlihat dari nilai recall yang masih sekitar 49%.

Pada dataset yang tidak seimbang, accuracy saja belum cukup untuk menilai performa model. Oleh karena itu, Precision, Recall, F1-Score, dan Confusion Matrix juga perlu diperhatikan. Keterbatasan dari praktikum ini adalah performa model masih dapat ditingkatkan agar lebih baik dalam mendeteksi pelanggan yang berpotensi churn.